# H5 Pre-Registered Replication: `Llama-3.1-8B-Instruct` (`fp16`, 2xT4)

**Checklist before running:**
1. `docs/llama31_trace_source_prereg.md` is committed to `main`.
2. Kaggle Settings -> **Accelerator**: `GPU T4 x2`.
3. Kaggle Settings -> **Add-ons -> Secrets**: enable `HF_TOKEN` (with Meta Llama 3.1 access).
4. Attach dataset uploaded from `dist/kaggle_llama31_h5_bundle.zip` (mounted under `/kaggle/input/`).

In [ ]:
# Cell 1: Verify GPUs, ensure peft/accelerate, authenticate HF_TOKEN, and locate bundle
import glob, os, shutil, subprocess, sys
from pathlib import Path
for pkg in ('peft', 'accelerate'):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
import torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

assert torch.cuda.device_count() >= 2, f'Expected 2xT4 GPUs, got {torch.cuda.device_count()}'
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

login(token=UserSecretsClient().get_secret('HF_TOKEN'))

matches = glob.glob('/kaggle/input/**/bundle_manifest.json', recursive=True)
assert matches, 'Upload dist/kaggle_llama31_h5_bundle.zip as a Kaggle Dataset and attach it!'
BUNDLE_ROOT = Path(matches[0]).resolve().parents[1]
WORK = Path('/kaggle/working/h5_workspace')
WORK.mkdir(parents=True, exist_ok=True)
for sub in ('structured_llm', 'audit', 'docs', 'bundle'):
    dst = WORK / sub
    if not dst.exists():
        shutil.copytree(BUNDLE_ROOT / sub, dst)
RUN = WORK / 'run_llama31_h5'
print('Workspace ready at', WORK)

In [ ]:
# Cell 2: Generate strict Calc-clean self traces (K >= 200) & verify pre-training invariants
!PYTHONPATH={WORK} python -u {WORK}/audit/llama31_trace_source_20261127/hf_runner.py \
    gen-self --bundle {WORK}/bundle --run {RUN}

In [ ]:
# Cell 3: Train all 4 LoRA adapters ({self, human} x {seed 0, seed 1}, 200 steps each)
for seed in (0, 1):
    for src in ('self', 'human'):
        !PYTHONPATH={WORK} python -u {WORK}/audit/llama31_trace_source_20261127/hf_runner.py \
            train --run {RUN} --seed {seed} --src {src}

In [ ]:
# Cell 4: Evaluate & intervene on 500 clean GSM-Plus holdout items (resumable)
!PYTHONPATH={WORK} python -u {WORK}/audit/llama31_trace_source_20261127/hf_runner.py \
    eval --bundle {WORK}/bundle --run {RUN} --tag base --mode base_prefill

for seed in (0, 1):
    for src in ('self', 'human'):
        adp = f'{RUN}/adapters/trace_{src}_200_s{seed}'
        !PYTHONPATH={WORK} python -u {WORK}/audit/llama31_trace_source_20261127/hf_runner.py \
            eval --bundle {WORK}/bundle --run {RUN} --tag {src}_s{seed} --mode tuned --adapter {adp}

In [ ]:
# Cell 5: Run pre-registered H5 & H5b analysis and archive outputs for download
!PYTHONPATH={WORK} python -u {WORK}/audit/llama31_trace_source_20261127/hf_runner.py \
    analyze --run {RUN}

shutil.make_archive('/kaggle/working/results_llama31_h5', 'zip', RUN)
print('Done! Download /kaggle/working/results_llama31_h5.zip from the Output panel.')